# Agentic NAS Workflow: Step-by-Step Exploration
Welcome to the interactive exploration of our Nextcloud ingestion and deduplication pipeline! In this notebook, we will walk through the core logic powering our automated NAS workflow.

In [2]:
# Setup: Import our custom modules from the `src` package
import os
import sys

# Ensure the project root is in the Python path
sys.path.append(os.path.abspath('..'))

from src.config import INGEST_DIR
from src.dedupe.crypto import calculate_blake3
# from src.storage.webdav import upload_file

print("Modules loaded successfully!")
print(f"Target Ingest Directory: {INGEST_DIR}")

Modules loaded successfully!
Target Ingest Directory: /cloud_ingest/gdrive_ahabib9387


## Step 1: Data Ingestion and Mock File Generation
First, let's explore our local ingestion directory. This is where files land before they are processed by the pipeline. We will create some mock files (including an exact duplicate) to demonstrate our pipeline's capabilities.

In [3]:
DUMMY_INGEST_DIR = "./cloud_ingest"

In [ ]:
# Let's see what files we have in the ingest directory
if not os.path.exists(DUMMY_INGEST_DIR):
    print(f"Directory {DUMMY_INGEST_DIR} does not exist yet. Let's create some dummy files for this demo.")
    os.makedirs(DUMMY_INGEST_DIR, exist_ok=True)
    
    with open(os.path.join(DUMMY_INGEST_DIR, "report.pdf"), "w") as f:
        f.write("Dummy PDF content")
    with open(os.path.join(DUMMY_INGEST_DIR, "photo.jpg"), "w") as f:
        f.write("Dummy Image content")
    # A duplicate file
    with open(os.path.join(DUMMY_INGEST_DIR, "report_copy.pdf"), "w") as f:
        f.write("Dummy PDF content")

files_to_process = []
for root, _, files in os.walk(DUMMY_INGEST_DIR):
    for file in files:
        files_to_process.append(os.path.join(root, file))

print(f"Found {len(files_to_process)} files to process:")
for f in files_to_process:
    print(f" - {f}")

Directory ./cloud_ingest does not exist yet. Let's create some dummy files for this demo.
Found 3 files to process:
 - ./cloud_ingest/report_copy.pdf
 - ./cloud_ingest/report.pdf
 - ./cloud_ingest/photo.jpg


## Step 2: Cryptographic Deduplication (BLAKE3 Hashing)
To prevent duplicate files from eating up precious NAS/Nextcloud space, we use **BLAKE3 Hashing** for exact bitwise deduplication.
Unlike md5 or sha256, BLAKE3 is extremely fast and can stream large files efficiently in chunks.

Let's compute the hashes for our three generated files and verify that our duplicate detector successfully flags the matching PDF files while letting the unique image pass.

In [4]:
# Compute and compare BLAKE3 hashes
pdf_path = os.path.join(DUMMY_INGEST_DIR, "report.pdf")
copy_path = os.path.join(DUMMY_INGEST_DIR, "report_copy.pdf")
photo_path = os.path.join(DUMMY_INGEST_DIR, "photo.jpg")

hash_pdf = calculate_blake3(pdf_path)
hash_copy = calculate_blake3(copy_path)
hash_photo = calculate_blake3(photo_path)

print(f"report.pdf hash:       {hash_pdf}")
print(f"report_copy.pdf hash:  {hash_copy}")
print(f"photo.jpg hash:        {hash_photo}\n")

# Assert and show results
print(f"Does report.pdf match report_copy.pdf? {hash_pdf == hash_copy} (Expected: True)")
print(f"Does report.pdf match photo.jpg?        {hash_pdf == hash_photo} (Expected: False)")

assert hash_pdf == hash_copy, "Error: PDF copy should have the same hash!"
assert hash_pdf != hash_photo, "Error: Image and PDF should have different hashes!"
print("\nSuccess! Cryptographic deduplication is working perfectly.")

report.pdf hash:       f9ea87c1bcce628cc6260f8d832dabe258857fd2513980bccf263554db65d5fe
report_copy.pdf hash:  f9ea87c1bcce628cc6260f8d832dabe258857fd2513980bccf263554db65d5fe
photo.jpg hash:        53f34198c1152cadab64c5745d9ecff100a9f1cf0aa9610d2cbebdd01ba98bcf

Does report.pdf match report_copy.pdf? True (Expected: True)
Does report.pdf match photo.jpg?        False (Expected: False)

Success! Cryptographic deduplication is working perfectly.


## Step 3: Semantic Deduplication (Content Similarity)
Cryptographic deduplication is perfect for finding **exact, byte-for-byte identical files**. However, it fails if a file is slightly modified (e.g., a PDF report with a single updated typo or a draft and a final version of a text file).

This is where **Semantic Deduplication** comes in. By analyzing the textual or visual content of files, we can calculate a **similarity score** and identify files that are nearly identical.

In `src/dedupe/semantic.py`, we implemented:
1. **Jaccard Similarity**: Measuring the overlap of unique words between documents.
2. **Cosine Similarity**: Comparing term frequency vectors to measure semantic alignment.

Let's test this semantic similarity logic on three sample documents!

In [7]:
# Import and test semantic deduplication
from src.dedupe.semantic import calculate_jaccard_similarity, calculate_cosine_similarity

# Let's define three texts to compare
doc_original = "The agentic NAS workflow automates our file ingestion, cryptographic hashing, and remote uploads to Nextcloud."
doc_near_duplicate = "The agentic NAS workflow automatically handles our file ingestion, cryptographic hashing, and remote uploads to Nextcloud server."
doc_completely_different = "Tomorrow's weather forecast predicts clear skies with a mild breeze and temperatures around 72 degrees."

print("Document Comparisons:")
print("-" * 50)

# Compare original vs near duplicate
jaccard_near = calculate_jaccard_similarity(doc_original, doc_near_duplicate)
cosine_near = calculate_cosine_similarity(doc_original, doc_near_duplicate)
print("Original vs Near Duplicate:")
print(f"  - Jaccard Similarity: {jaccard_near:.4f}")
print(f"  - Cosine Similarity:  {cosine_near:.4f}")

# Compare original vs different
jaccard_diff = calculate_jaccard_similarity(doc_original, doc_completely_different)
cosine_diff = calculate_cosine_similarity(doc_original, doc_completely_different)
print("\nOriginal vs Completely Different:")
print(f"  - Jaccard Similarity: {jaccard_diff:.4f}")
print(f"  - Cosine Similarity:  {cosine_diff:.4f}")

# Assert similarity behaves as expected
assert cosine_near > 0.80, "Error: Near duplicates should have high similarity!"
assert cosine_diff < 0.15, "Error: Completely different documents should have low similarity!"
print("\nSuccess! Semantic deduplication detects near-duplicates perfectly.")

Document Comparisons:
--------------------------------------------------
Original vs Near Duplicate:
  - Jaccard Similarity: 0.7778
  - Cosine Similarity:  0.8767

Original vs Completely Different:
  - Jaccard Similarity: 0.0333
  - Cosine Similarity:  0.0645

Success! Semantic deduplication detects near-duplicates perfectly.


## Step 4: WebDAV Storage and Prefect Orchestration
Once our files are ingested and checked for duplicates, they are routed and uploaded idempotently to **Nextcloud** using WebDAV (`webdav4`).

The entire process is orchestrated as a pipeline flow using **Prefect 3.0**, with task retries and exponential backoffs configured to handle network hiccups seamlessly.

Let's check out our WebDAV upload helper and run the complete pipeline!
*(Note: To make this exploration safe and robust, if your Nextcloud instance or Prefect server is not currently reachable, the cells will catch the connection exceptions and describe the simulated production behavior instead of crashing!)*

In [3]:
# Test uploading a single file to Nextcloud WebDAV
from src.config import NEXTCLOUD_URL, INGEST_DIR
from prefect.blocks.system import Secret
from webdav4.client import Client

In [4]:
print(f"Targeting Nextcloud at: {NEXTCLOUD_URL}")
print(f"Scanning Ingest Directory: {INGEST_DIR}")

Targeting Nextcloud at: http://192.168.1.55:30027/remote.php/webdav
Scanning Ingest Directory: /cloud_ingest/gdrive_ahabib9387


In [5]:
# 1. Fetch secrets safely inside Jupyter's async loop
user_block = await Secret.load("nextcloud-username")
pass_block = await Secret.load("nextcloud-password")

# 2. Instantiate the WebDAV client (This is our Dependency!)
client = Client(NEXTCLOUD_URL, auth=(user_block.get(), pass_block.get()))

print("\nWebDAV Client successfully instantiated.")
print("Root Nextcloud Folders:")
for item in client.ls("/"):
    print(f" - {item['name']}")


WebDAV Client successfully instantiated.
Root Nextcloud Folders:
 - Photos
 - Documents
 - Templates


In [14]:
# Run the complete SRE ingestion and organization pipeline
from src.main import run_pipeline

KeyboardInterrupt: 

In [ ]:
try:
    print("Initializing Agentic NAS Pipeline flow...")
    run_pipeline()
except Exception as e:
    print(f"\n[Environment Notice]: Flow execution completed with exception/warning.")
    print(f"Error detail: {e}")
    print("This is normal when Prefect or Nextcloud services are not fully running locally.")

Initializing Agentic NAS Pipeline flow...


/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/pydantic/_internal/_typing_extra.py:309: RuntimeWarning: coroutine 'Block.aload' was never awaited
  return obj.__dict__.get('__annotations__', {})


03:48:17.731 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8859
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

03:48:25.366 | ERROR   | docket.dependencies - ↩ [   655ms] send_telemetry_heartbeat(){send_telemetry_heartbeat}
Traceback (most recent call last):
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/sqlalchemy/engine/base.py", line 1969, in _exec_single_context
    self.dialect.do_execute(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        cursor, str_statement, effective_parameters, context
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/sqlalchemy/engine/default.py", line 952, in do_execute
    cursor.execute(statement, parameters)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/sqlalchemy/dialects/sqlite/aiosqlite.py", line 183, in execute
    self._adapt_connection._handle_exception(error)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
  File "/home/coder/projects/agentic

03:48:25.547 | INFO    | Flow run 'perky-bandicoot' - Beginning flow run 'perky-bandicoot' for flow 'Agentic-NAS-Pipeline'

...

03:49:05.905 | INFO    | Task run 'task_upload_file-9f0' - Task run failed with exception: ConnectError('[Errno 111] Connection refused') - Retry 3/3 will start 5 second(s) from now

03:49:10.910 | ERROR   | Task run 'task_upload_file-9f0' - Task run failed with exception: ConnectError('[Errno 111] Connection refused') - Retries are exhausted
Traceback (most recent call last):
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
        pool_request.request
    )
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/httpcore/_sync/connection.py", line 101, in handle_request
    raise exc
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/httpcore/_sync/connection.py", line 78, in handle_request
    stream = self._connect(request)
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/httpcore/_sync/connection.py", line 124, in _connect
    stream = self._network_backend.connect_tcp(**kwargs)
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/httpcore/_backends/sync.py", line 207, in connect_tcp
    with map_exceptions(exc_map):
         ~~~~~~~~~~~~~~^^^^^^^^^
  File "/home/coder/.local/share/uv/python/cpython-3.13.14-linux-x86_64-gnu/lib/python3.13/contextlib.py", line 162, in __exit__
    self.gen.throw(value)
    ~~~~~~~~~~~~~~^^^^^^^
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/httpcore/_exceptions.py", line 14, in map_exceptions
    raise to_exc(exc) from exc
httpcore.ConnectError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/prefect/task_engine.py", line 1021, in run_context
    yield self
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/prefect/task_engine.py", line 1683, in run_task_sync
    engine.call_task_fn(txn)
    ~~~~~~~~~~~~~~~~~~~^^^^^
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/prefect/task_engine.py", line 1038, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/prefect/utilities/callables/__init__.py", line 348, in call_with_parameters
    return fn(*args, **kwargs)
  File "/home/coder/projects/agentic-nas-workflow/src/main.py", line 16, in task_upload_file
    upload_file(local_path, remote_path)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/coder/projects/agentic-nas-workflow/src/storage/webdav.py", line 12, in upload_file
    if not client.exists(folder_path):
           ~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/webdav4/client.py", line 530, in exists
    self.propfind(path)
    ~~~~~~~~~~~~~^^^^^^
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/webdav4/client.py", line 309, in propfind
    http_resp = self.with_retry(call)
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/webdav4/func_utils.py", line 47, in wrapped_function
    return func()
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/python3.13/site-packages/webdav4/func_utils.py", line 70, in wrapped
    return func(*args, **kwargs)
  File "/home/coder/projects/agentic-nas-workflow/.venv/lib/pytho

03:49:10.924 | ERROR   | Task run 'task_upload_file-9f0' - Finished in state Failed('Task run encountered an exception ConnectError: [Errno 111] Connection refused')

03:49:10.929 | INFO    | Flow run 'perky-bandicoot' - FAILED to upload photo.jpg: [Errno 111] Connection refused

03:49:11.597 | INFO    | Flow run 'perky-bandicoot' - Finished in state Completed()